# Experiment 1: summarize available approaches
Discover completed recordings from metadata and summarize each task independently. Set `tasks=None` for discovery or specify a list. Checkpoint/protocol validation remains strict. Evaluation seeds are not training seeds.


In [1]:
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Find this checkout whether Jupyter starts in the repository or notebooks directory.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks" / "exp1_analysis.py").exists())
sys.path.insert(0, str(REPO / "notebooks"))
import exp1_analysis as exp1
# Refresh helper changes even when this Jupyter kernel was already running.
import importlib
exp1 = importlib.reload(exp1)

from b1z1_paper_names import DISPLAY_NAMES, LATEX_METHOD_NAMES
# Paper labels differ from implementation ablation numbers; task IDs stay unchanged.
CONFIG = {
    # Figure typography in points; requires installed Times New Roman.
    "figure_title_font_size": 12,
    "title_font_size": 11,
    "axis_label_font_size": 10,
    "tick_font_size": 9,
    "legend_font_size": 9,
    "annotation_font_size": 8,
    "input_roots": [str(REPO / "evaluation/exp1")],  # Recursively inspect this experiment folder.
    "task": None,  # Legacy single-task mode uses output_dir directly.
    "tasks": None,  # Or ["b1z1_pact", "b1z1_unifp_original", "b1z1_ppo_pos"].
    "display_names": dict(DISPLAY_NAMES),
    "task_options": {},  # Per task: {"checkpoint_sha256": "...", "display_name": "..."}.
    "checkpoint_sha256": None,  # Optional exact hash from metadata.json; otherwise require a unique hash.
    "ignore_evaluator_source_hash": True,  # Opt in to ignore evaluator-code hashes ONLY; settings/checkpoints remain strict.
    "expected_seeds": [0, 1, 2],
    "output_dir": str(REPO / "analysis/exp1"),  # Batch output: <output_dir>/<task>/.
    "histogram_edges": np.linspace(-0.5, 0.5, 101).tolist(),  # Metres; reuse across approaches.
    "histogram_frame": "yaw",  # Successor base-yaw frame; "world" is also supported.
    "include_combined": True,  # Explicitly opt in to pooled stationary+locomotion trials.
    "acceleration_metrics": True,  # Backward differences, no filtering, no reset/gap crossings.
}
# Optional JSON override for reproducible batch execution/testing.
if os.environ.get("EXP1_ANALYSIS_CONFIG"):
    CONFIG.update(json.loads(Path(os.environ["EXP1_ANALYSIS_CONFIG"]).read_text()))
CONFIG


{'input_roots': ['/home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1'],
 'task': None,
 'tasks': None,
 'display_names': {'b1z1_pact': 'PILOT',
  'b1z1_ppo_pos': 'Pos',
  'b1z1_unifp_original': 'UniFP',
  'b1z1_pact_ab4_coupled_none': 'A4',
  'b1z1_pact_ab5_coupled_concat': 'A5',
  'b1z1_pact_ab6_position_film': 'A1',
  'b1z1_pact_ab7_position_full': 'A2',
  'b1z1_pact_ab8_coupled_film': 'A3',
  'b1z1_pact_ab9_representation': 'A6',
  'b1z1_pact_ab10_actor': 'A7'},
 'task_options': {},
 'checkpoint_sha256': None,
 'ignore_evaluator_source_hash': True,
 'expected_seeds': [0, 1, 2],
 'output_dir': '/home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/analysis/exp1',
 'histogram_edges': [-0.5,
  -0.49,
  -0.48,
  -0.47,
  -0.46,
  -0.45,
  -0.44,
  -0.43,
  -0.42,
  -0.41000000000000003,
  -0.4,
  -0.39,
  -0.38,
  -0.37,
  -0.36,
  -0.35,
  -0.33999999999999997,
  -0.32999999999999996,
  -0.32,
  -0.31,
  -0.3,
  -0.29000000

## Definitions
Event recovery is also evaluated separately for EE-position tolerance and for planar velocity plus yaw-rate tolerances. The eligible-event denominator, response window, contiguous dwell and whole-trial failure censoring are unchanged. Per-event outcomes and per-seed recovered counts are saved; each seed's recovery fraction is averaged equally across seeds. Regenerate summaries to add these metrics.
The additional `S_track_velocity` requires planar velocity **and** yaw-rate tolerances, while `S_track_manipulation` requires only the EE-position tolerance. Each uses the full prescribed horizon, with zero credit for failed/missing steps and only its own errors required finite. These are computed from transitions, then averaged equally across trials and evaluation seeds. Rerun this notebook for all methods before comparing the new scores.
Tracking errors use finite pre-failure samples. Each trial contributes equally within its evaluation seed; seeds then contribute equally. `S_track` uses the full prescribed horizon, including zero credit after failure. SD is the sample SD (`ddof=1`); one seed has undefined SD.

Physical-limit percentages pool valid joint-substep numerators/denominators within each seed. Event peaks and integrals require complete response windows. Any physical failure censors all recovery events in that trial, matching the evaluator. Recovery times are averaged only over recovered events; no missing value becomes zero.

In [2]:
from experiment_batch import plan_summaries
plans, inventory = plan_summaries(CONFIG, experiment=1)
display(pd.DataFrame(inventory))  # Tasks, checkpoints, seeds and output destinations.
summaries = {}
for index, settings in enumerate(plans, 1):
    print(f"[{index}/{len(plans)}] {settings['task']}")
    summary = exp1.summarize(settings)  # Preserve strict per-method validation/statistics.
    summaries[settings["task"]] = summary
    display(pd.DataFrame(summary["aggregates"]))
    print("\n".join(summary["diagnostics"]) or "No missing-data diagnostics.")
    print("Exported:", settings["output_dir"])


,task,runs,evaluation_seeds,checkpoints,output_dir
0,b1z1_pact,9,"[0, 1, 2]",[2eaf0240106b78e6423f2d6e310318d72b1da561c1af3...,/home/oyoungquist/Research/Genesis_Development...
1,b1z1_pact_ab10_actor,9,"[0, 1, 2]",[0674cc16b851e629242ea4e83e913014e9d80d35254a9...,/home/oyoungquist/Research/Genesis_Development...
2,b1z1_pact_ab4_coupled_none,9,"[0, 1, 2]",[acf4a8e0819d75a4b7c3884f3bc95965d550037d5d064...,/home/oyoungquist/Research/Genesis_Development...
3,b1z1_pact_ab5_coupled_concat,9,"[0, 1, 2]",[66955da0c81ae8593487ab2feb192a3b8c2e137fe2c81...,/home/oyoungquist/Research/Genesis_Development...
4,b1z1_pact_ab6_position_film,9,"[0, 1, 2]",[128d2859a46a902a141968f607fa04df39ba90658304f...,/home/oyoungquist/Research/Genesis_Development...
5,b1z1_pact_ab7_position_full,9,"[0, 1, 2]",[46943a2d8315e04c48700b6a5223d860fcaf955e43ae5...,/home/oyoungquist/Research/Genesis_Development...
6,b1z1_pact_ab8_coupled_film,9,"[0, 1, 2]",[928a3dd041e64b5da76ea9b1120ebd39062ea41a61b38...,/home/oyoungquist/Research/Genesis_Development...
7,b1z1_pact_ab9_representation,9,"[0, 1, 2]",[74bab1b60b92bf9fdfda43dab84c44f0690ad84577d01...,/home/oyoungquist/Research/Genesis_Development...
8,b1z1_ppo_pos,9,"[0, 1, 2]",[5b08e70fb898608e3a152bb39c9dc03a03baf6eade217...,/home/oyoungquist/Research/Genesis_Development...
9,b1z1_unifp_original,9,"[0, 1, 2]",[3950bf33b7112f11a01fbf3f973a87ddf5d34abaf34d0...,/home/oyoungquist/Research/Genesis_Development...


[1/10] b1z1_pact


/tmp/ipykernel_1152865/4278802012.py:7: RuntimeWarning: Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
  summary = exp1.summarize(settings)  # Preserve strict per-method validation/statistics.


,condition,load_range,motion,metric,unit,mean,sd,contributing_seeds,attempted_trials,contributing_trials
0,combined-disturbance,in,combined,ee_error_m,m,0.053892,0.000441,3,300,300.0
1,combined-disturbance,in,combined,planar_velocity_error_m_s,m/s,0.081110,0.001906,3,300,300.0
2,combined-disturbance,in,combined,yaw_error_rad_s,rad/s,0.111316,0.001775,3,300,300.0
3,combined-disturbance,in,combined,S_track,fraction,0.787209,0.014197,3,300,300.0
4,combined-disturbance,in,combined,failure_rate,fraction,0.003333,0.005774,3,300,300.0
...,...,...,...,...,...,...,...,...,...,...
157,nominal,in,stationary,recovery_fraction,fraction,NaN,NaN,0,150,NaN
158,nominal,in,stationary,recovery_fraction_ee,fraction,NaN,NaN,0,150,NaN
159,nominal,in,stationary,recovery_fraction_velocity,fraction,NaN,NaN,0,150,NaN
160,nominal,in,stationary,torso_angular_acceleration_rms,rad/s^2,7.461505,0.017808,3,150,NaN


Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
Exported: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/analysis/exp1/b1z1_pact
[2/10] b1z1_pact_ab10_actor


...

,condition,load_range,motion,metric,unit,mean,sd,contributing_seeds,attempted_trials,contributing_trials
0,combined-disturbance,in,combined,ee_error_m,m,0.063870,0.001511,3,300,300.0
1,combined-disturbance,in,combined,planar_velocity_error_m_s,m/s,0.064203,0.000780,3,300,300.0
2,combined-disturbance,in,combined,yaw_error_rad_s,rad/s,0.086230,0.000435,3,300,300.0
3,combined-disturbance,in,combined,S_track,fraction,0.771631,0.012970,3,300,300.0
4,combined-disturbance,in,combined,failure_rate,fraction,0.016667,0.005774,3,300,300.0
...,...,...,...,...,...,...,...,...,...,...
157,nominal,in,stationary,recovery_fraction,fraction,NaN,NaN,0,150,NaN
158,nominal,in,stationary,recovery_fraction_ee,fraction,NaN,NaN,0,150,NaN
159,nominal,in,stationary,recovery_fraction_velocity,fraction,NaN,NaN,0,150,NaN
160,nominal,in,stationary,torso_angular_acceleration_rms,rad/s^2,6.934971,0.042197,3,150,NaN


Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
Exported: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/analysis/exp1/b1z1_ppo_pos
[10/10] b1z1_unifp_original


/tmp/ipykernel_1152865/4278802012.py:7: RuntimeWarning: Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
  summary = exp1.summarize(settings)  # Preserve strict per-method validation/statistics.


,condition,load_range,motion,metric,unit,mean,sd,contributing_seeds,attempted_trials,contributing_trials
0,combined-disturbance,in,combined,ee_error_m,m,0.071543,0.000873,3,300,300.0
1,combined-disturbance,in,combined,planar_velocity_error_m_s,m/s,0.063496,0.000746,3,300,300.0
2,combined-disturbance,in,combined,yaw_error_rad_s,rad/s,0.090593,0.001154,3,300,300.0
3,combined-disturbance,in,combined,S_track,fraction,0.768178,0.010794,3,300,300.0
4,combined-disturbance,in,combined,failure_rate,fraction,0.060000,0.026458,3,300,300.0
...,...,...,...,...,...,...,...,...,...,...
157,nominal,in,stationary,recovery_fraction,fraction,NaN,NaN,0,150,NaN
158,nominal,in,stationary,recovery_fraction_ee,fraction,NaN,NaN,0,150,NaN
159,nominal,in,stationary,recovery_fraction_velocity,fraction,NaN,NaN,0,150,NaN
160,nominal,in,stationary,torso_angular_acceleration_rms,rad/s^2,4.631631,0.012261,3,150,NaN


Evaluator-source hash validation disabled by ignore_evaluator_source_hash=True; evaluation-code equivalence has not been verified. Original hashes are retained in provenance.
Exported: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/analysis/exp1/b1z1_unifp_original


## Portable exports
Exact equal-seed weighted box statistics (5/25/50/75/95 percentiles) are also exported in `summary.json`. They include all valid errors, including histogram tails. Rerun this notebook for older summaries that lack `ee_boxplots`.
`summary.json` contains provenance, protocols, per-trial/event results, per-seed values, aggregate statistics and diagnostics. `summary.csv` contains aggregate rows; `ee_histograms.csv` contains per-seed counts and explicit underflow/overflow denominators. `COMPLETE` is written only after all exports succeed.

Histograms use signed measured-minus-reference errors, in metres, rotated with the matching successor orientation. Counts are normalized per seed using **all valid samples**, including tails, before seed averaging. Failed tails are never padded.

In [3]:
for settings in plans:
    print(settings["task"])
    histograms = pd.read_csv(Path(settings["output_dir"]) / "ee_histograms.csv")
    tails = histograms.drop_duplicates(["seed", "condition", "load_range", "motion", "axis"])
    display(tails[["seed", "condition", "load_range", "motion", "axis", "valid_samples", "underflow", "overflow"]])


b1z1_pact


,seed,condition,load_range,motion,axis,valid_samples,underflow,overflow
0,0,combined-disturbance,in,stationary,x,37500,0,0
100,0,combined-disturbance,in,stationary,y,37500,0,0
200,0,combined-disturbance,in,stationary,z,37500,0,5
300,0,combined-disturbance,in,locomotion,x,37163,0,0
400,0,combined-disturbance,in,locomotion,y,37163,0,0
500,0,combined-disturbance,in,locomotion,z,37163,0,0
600,0,combined-disturbance,in,combined,x,74663,0,0
700,0,combined-disturbance,in,combined,y,74663,0,0
800,0,combined-disturbance,in,combined,z,74663,0,5
900,1,combined-disturbance,in,stationary,x,37500,0,0


b1z1_pact_ab10_actor


,seed,condition,load_range,motion,axis,valid_samples,underflow,overflow
0,0,combined-disturbance,in,stationary,x,37500,0,0
100,0,combined-disturbance,in,stationary,y,37500,0,0
200,0,combined-disturbance,in,stationary,z,37500,0,0
300,0,combined-disturbance,in,locomotion,x,37500,4,0
400,0,combined-disturbance,in,locomotion,y,37500,0,0
500,0,combined-disturbance,in,locomotion,z,37500,0,0
600,0,combined-disturbance,in,combined,x,75000,4,0
700,0,combined-disturbance,in,combined,y,75000,0,0
800,0,combined-disturbance,in,combined,z,75000,0,0
900,1,combined-disturbance,in,stationary,x,37500,0,0


b1z1_pact_ab4_coupled_none


...

,seed,condition,load_range,motion,axis,valid_samples,underflow,overflow
0,0,combined-disturbance,in,stationary,x,37117,0,0
100,0,combined-disturbance,in,stationary,y,37117,0,0
200,0,combined-disturbance,in,stationary,z,37117,0,0
300,0,combined-disturbance,in,locomotion,x,37500,1,0
400,0,combined-disturbance,in,locomotion,y,37500,0,0
500,0,combined-disturbance,in,locomotion,z,37500,0,0
600,0,combined-disturbance,in,combined,x,74617,1,0
700,0,combined-disturbance,in,combined,y,74617,0,0
800,0,combined-disturbance,in,combined,z,74617,0,0
900,1,combined-disturbance,in,stationary,x,37500,0,0


b1z1_ppo_pos


,seed,condition,load_range,motion,axis,valid_samples,underflow,overflow
0,0,combined-disturbance,in,stationary,x,37500,0,0
100,0,combined-disturbance,in,stationary,y,37500,0,0
200,0,combined-disturbance,in,stationary,z,37500,0,0
300,0,combined-disturbance,in,locomotion,x,36915,4,0
400,0,combined-disturbance,in,locomotion,y,36915,0,0
500,0,combined-disturbance,in,locomotion,z,36915,0,0
600,0,combined-disturbance,in,combined,x,74415,4,0
700,0,combined-disturbance,in,combined,y,74415,0,0
800,0,combined-disturbance,in,combined,z,74415,0,0
900,1,combined-disturbance,in,stationary,x,36916,0,0


b1z1_unifp_original


,seed,condition,load_range,motion,axis,valid_samples,underflow,overflow
0,0,combined-disturbance,in,stationary,x,35849,198,0
100,0,combined-disturbance,in,stationary,y,35849,0,0
200,0,combined-disturbance,in,stationary,z,35849,0,0
300,0,combined-disturbance,in,locomotion,x,36003,162,0
400,0,combined-disturbance,in,locomotion,y,36003,3,0
500,0,combined-disturbance,in,locomotion,z,36003,0,0
600,0,combined-disturbance,in,combined,x,71852,360,0
700,0,combined-disturbance,in,combined,y,71852,3,0
800,0,combined-disturbance,in,combined,z,71852,0,0
900,1,combined-disturbance,in,stationary,x,36162,182,0
